# ClinVar — the crowd-sourced clinical truth set

A predictor's number is meaningless until you compare it against a **truth set** — variants whose clinical status someone already established. **ClinVar** is the public archive of clinical assertions from many labs. This data is **REAL** (genuine CFTR ClinVar assertions); every row reads `source == 'REAL'`.

> ✅ **REAL data.** These are genuine ClinVar assertions for *CFTR*, fetched live by the cell below. benchmark/01 adds the second truth set (CFTR2); the archived integration notebook cross-checks the two.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## Fetching the REAL data — NCBI's ClinVar FTP release, with version control

There is no per-gene ClinVar API to query — NCBI publishes ClinVar as one big
tab-delimited dump, `variant_summary.txt.gz` (~440 MB, updated **~weekly**), at
`ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/`. The cell below **streams and
filters it on the fly** (`GeneSymbol == 'CFTR'` and `Assembly == 'GRCh38'` —
ClinVar lists most variants under both GRCh37 *and* GRCh38, so skipping the
assembly filter would double-count), keeping every row (not just missense —
see §1) and six columns: `VariationID` (ClinVar's own stable ID), `Name`,
`Type` (SNV/indel/…), `ClinicalSignificance`, `ClinSigSimple` (ClinVar's own
simplified flag), and `ReviewStatus`.

**Version control.** The rolling `variant_summary.txt.gz` has no version number
in its filename, but it isn't untracked either:

- **Default (`CLINVAR_RELEASE = "latest"`):** the cell issues a `HEAD` request
  first and records the response's `Last-Modified` timestamp — a real,
  checkable version stamp — into `data/clinvar_cftr.release.json` alongside
  the extract. `load_clinvar()` exposes it as the `clinvar_release` column, so
  every table downstream carries its own provenance.
- **Reproducing a past run:** NCBI also publishes **monthly dated snapshots**
  back to 2014, at `.../archive/variant_summary_YYYY-MM.txt.gz` (2025 onward)
  or `.../archive/<year>/variant_summary_YYYY-MM.txt.gz` (2014–2024). Set
  `CLINVAR_RELEASE = "2026-03"` (for example) instead of `"latest"` to pull
  that exact month's file — the cell resolves the right archive path for you.

ClinVar is public domain (CC0 / U.S. Government work) — no license
restriction, attribution requested.

In [2]:
import requests, gzip, json
from datetime import datetime, timezone

DATA_DIR = pathlib.Path.cwd().parent / "data"
CLINVAR_TSV = DATA_DIR / "clinvar_cftr.tsv"
CLINVAR_RELEASE_JSON = DATA_DIR / "clinvar_cftr.release.json"

# "latest" pulls the current rolling file, dated via its HTTP Last-Modified header.
# To reproduce a past run, set this to a specific "YYYY-MM" instead (e.g. "2026-03")
# -- see the markdown above for what that resolves to.
CLINVAR_RELEASE = "latest"


def clinvar_url(release: str) -> str:
    """Resolve a ClinVar release label to its download URL.

    'latest'  -> the always-current rolling file (no version in the filename).
    'YYYY-MM' -> NCBI's monthly dated archive. Archive layout changed over time:
                 2025 onward the file sits directly in archive/; 2014-2024 it's
                 nested one level deeper under archive/<year>/.
    """
    base = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited"
    if release == "latest":
        return f"{base}/variant_summary.txt.gz"
    year = int(release[:4])
    if year <= 2024:
        return f"{base}/archive/{year}/variant_summary_{release}.txt.gz"
    return f"{base}/archive/variant_summary_{release}.txt.gz"


def fetch_clinvar_cftr(release: str):
    """Stream-filter one ClinVar variant_summary release to CFTR/GRCh38 (REAL)."""
    url = clinvar_url(release)
    if release == "latest":
        head = requests.head(url, timeout=20)
        head.raise_for_status()
        resolved_version = head.headers.get("Last-Modified", "unknown")
    else:
        resolved_version = release   # the pinned month IS the version, no HEAD needed

    resp = requests.get(url, stream=True, timeout=30)
    resp.raise_for_status()
    resp.raw.decode_content = False
    gz = gzip.GzipFile(fileobj=resp.raw)

    header = gz.readline().decode().rstrip("\n").split("\t")
    gene_idx, asm_idx = header.index("GeneSymbol"), header.index("Assembly")
    rows = [f for line in gz
            if (f := line.decode(errors="replace").rstrip("\n").split("\t"))[gene_idx] == "CFTR"
            and f[asm_idx] == "GRCh38"]
    return pd.DataFrame(rows, columns=header), resolved_version


if CLINVAR_TSV.exists():
    print(f"already fetched -> {CLINVAR_TSV.name} "
          f"(delete it and {CLINVAR_RELEASE_JSON.name} to re-fetch)")
else:
    DATA_DIR.mkdir(exist_ok=True)
    print(f"streaming ClinVar variant_summary.txt.gz (release={CLINVAR_RELEASE!r}, "
          f"~440 MB) and filtering to CFTR... ~40s")
    cftr, resolved_version = fetch_clinvar_cftr(CLINVAR_RELEASE)
    print(f"CFTR/GRCh38 rows: {len(cftr):,}  (resolved version: {resolved_version})")

    cols = ["VariationID", "Name", "Type", "ClinicalSignificance", "ClinSigSimple", "ReviewStatus"]
    cftr[cols].to_csv(CLINVAR_TSV, sep="\t", index=False)
    CLINVAR_RELEASE_JSON.write_text(json.dumps({
        "requested_release": CLINVAR_RELEASE,
        "resolved_version": resolved_version,
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "row_count": len(cftr),
        "source_url": clinvar_url(CLINVAR_RELEASE),
    }, indent=2))
    print(f"wrote {CLINVAR_TSV.relative_to(DATA_DIR.parent)} and {CLINVAR_RELEASE_JSON.name}")

already fetched -> clinvar_cftr.tsv (delete it and clinvar_cftr.release.json to re-fetch)


## 1 · ClinVar — what is it?

[ClinVar](https://www.ncbi.nlm.nih.gov/clinvar/) is a free, public archive run by the NIH. Clinical genetics labs around the world **submit their interpretation** of a variant: is it disease-causing or not? ClinVar aggregates those submissions and reports a clinical significance, usually one of:

- **Pathogenic** / **Likely pathogenic**
- **Uncertain significance** (a *VUS* — Variant of Uncertain Significance)
- **Likely benign** / **Benign**
- **Conflicting classifications** (labs disagreed)

Think of it as a giant, crowd-sourced ledger of *clinical opinion*. Its great strength is **breadth** — the cell above retrieves *every* CFTR record ClinVar has for GRCh38 (exact count printed there; usually 6,000+, and it grows over time — see the version-control note above). Its weakness is that it is **heterogeneous**: different submitters, different rigor, different years. Let's load it.

`load_clinvar()` itself is a thin reader in `toolkit.py`: it reads
`data/clinvar_cftr.tsv` (fetched by the cell above) and returns **every** row —
missense, nonsense, indels, splice, everything. `protein_variant` is only
populated for the subset that reduces to a simple single-residue missense
change (`''` otherwise), so you can filter to missense-only downstream without
losing the rest of the data by default. Ref: Landrum et al., *Nucleic Acids
Res*.

In [3]:
clinvar = tk.load_clinvar()
print('rows:', len(clinvar))
print('source values:', clinvar['source'].unique(), ' <- REAL data')
print('release:', clinvar['clinvar_release'].iloc[0])
print('with a resolvable missense key:', (clinvar['protein_variant'] != '').sum())
clinvar.head()

rows: 6165
source values: ['REAL']  <- REAL data
release: Tue, 04 Aug 2026 19:58:34 GMT
with a resolvable missense key: 2961


,variation_id,hgvs_name,protein_variant,clinvar_sig,clinsig_simple,variant_type,review_status,clinvar_release,source
0,7105,NM_000492.3(CFTR):c.1521_1523del (p.Phe508del),,Pathogenic,1,Deletion,practice guideline,"Tue, 04 Aug 2026 19:58:34 GMT",REAL
1,7106,NM_000492.4(CFTR):c.1516ATC[1] (p.Ile507del),,Pathogenic,1,Microsatellite,practice guideline,"Tue, 04 Aug 2026 19:58:34 GMT",REAL
2,7107,NM_000492.4(CFTR):c.1477C>T (p.Gln493Ter),Q493*,Pathogenic,1,single nucleotide variant,reviewed by expert panel,"Tue, 04 Aug 2026 19:58:34 GMT",REAL
3,7108,NM_000492.4(CFTR):c.328G>C (p.Asp110His),D110H,Pathogenic; drug response,1,single nucleotide variant,reviewed by expert panel,"Tue, 04 Aug 2026 19:58:34 GMT",REAL
4,7109,NM_000492.4(CFTR):c.350G>A (p.Arg117His),R117H,Pathogenic,1,single nucleotide variant,practice guideline,"Tue, 04 Aug 2026 19:58:34 GMT",REAL


Every row is `source == 'REAL'`: these are genuine ClinVar assertions for *CFTR*, not something the toolkit made up. Each row has:
- `variation_id` — ClinVar's own stable identifier for the variant
- `protein_variant` — the amino-acid change (e.g. `G551D`), `''` if this variant doesn't reduce to one
- `clinvar_sig` — the **raw** clinical significance text, exactly as ClinVar reports it — no collapsing
- `clinsig_simple` — ClinVar's **own** simplified flag (not derived by this toolkit): `1` = at least one current Pathogenic/Likely pathogenic (or risk-allele) submission, `0` = no such submission, `-1` = no clinical significance data at all. It is not a 3-way split — `0` covers true Benign, VUS, and "not provided" alike.
- `variant_type` — ClinVar's structural type (`single nucleotide variant`, `Deletion`, `Indel`, …)
- `review_status` — **how much confidence** to place in it (next section)
- `clinvar_release` — the version this extract was fetched at (see the fetch cell above)

## 2 · Review status — the gold stars (⭐) that matter most

**This is the single most important slide in the notebook.** Not every ClinVar assertion deserves equal trust. ClinVar attaches a **review status**, which maps to a **0–4 gold-star** rating:

| Stars | Review status | Roughly means |
|:--:|---|---|
| 0 ⭐ | *no assertion criteria provided* | someone's opinion, no stated method |
| 1 ⭐ | *criteria provided, single submitter* | one lab, with a method |
| 1 ⭐ | *criteria provided, conflicting classifications* | labs disagree |
| 2 ⭐ | *criteria provided, multiple submitters, no conflicts* | several labs agree |
| 3 ⭐ | *reviewed by expert panel* | e.g. the CFTR2 / ClinGen expert panel |
| 4 ⭐ | *practice guideline* | encoded in clinical practice guidelines |

Let's count how our CFTR variants are distributed across these levels.

In [4]:
clinvar['review_status'].value_counts()

review_status
criteria provided, single submitter                     3189
criteria provided, multiple submitters, no conflicts    1732
criteria provided, conflicting classifications           395
reviewed by expert panel                                 370
no assertion criteria provided                           351
no classification provided                                92
practice guideline                                        23
-                                                          9
no classification for the single variant                   4
Name: count, dtype: int64

**Read this carefully.** The largest bucket is *single submitter* (1 star) — one lab's call. Only a couple hundred are *expert panel* (3 stars), and a dozen are *practice guideline* (4 stars).

> **KEY LESSON:** a **1-star VUS** and a **3-star expert-panel Pathogenic** are *not* the same quality of evidence, even though both are 'in ClinVar'. When you build a benchmark, you often **filter to ≥2 stars** so a predictor is judged against assertions people actually trust — not against one lab's unreviewed guess. Throwing every star level into the same pile is one of the most common ways people accidentally lie to themselves with ClinVar.

## 3 · ClinVar's own significance signals — no collapsing

This notebook does not reduce `ClinicalSignificance` down to a simplified
pathogenic/benign/uncertain call. Collapsing categorical clinical text is
trickier than it looks: a few values, like `'not provided'` or
`'drug response'`, are not pathogenicity assessments at all, and folding them
into "uncertain" would conflate *no interpretation was given* with *assessed
and found uncertain* — two very different situations for anyone building a
benchmark on top of this data.

What you get instead is ClinVar's own two signals, unmodified:

- **`clinvar_sig`** — the raw text. It is *not* free-form: for CFTR it takes
  only a handful of distinct values (counted below), because ClinVar submitters
  work from the same ACMG-style vocabulary. A few are simply compound
  (`'Pathogenic/Likely pathogenic'`) or not a pathogenicity call at all
  (`'not provided'`, `'drug response'`).
- **`clinsig_simple`** — ClinVar's **own** simplified flag (see §1's table
  above): `1` if the variant has at least one current Pathogenic/Likely
  pathogenic submission, `0` if not, `-1` if there is no clinical significance
  data at all. This is an official ClinVar column, not something this toolkit
  computed — but note it is *binary*, not 3-way: `0` covers true Benign, VUS,
  and "not provided" without distinguishing them.

If a downstream analysis needs its own pathogenic/benign/uncertain split, build
it deliberately from one of these two columns and say so explicitly — don't
reach for a black-box helper.

In [5]:
print('clinvar_sig (raw, unmodified):')
print(clinvar['clinvar_sig'].value_counts().to_string())

print('\nclinsig_simple (ClinVar\'s own flag: 1=has a pathogenic-like call, 0=no, -1=no data):')
print(clinvar['clinsig_simple'].value_counts().to_string())

n_vus = int((clinvar['clinvar_sig'] == 'Uncertain significance').sum())
print(f"\nRows explicitly marked 'Uncertain significance': {n_vus:,} "
      f"({100*n_vus/len(clinvar):.0f}% of all rows). 'not provided' and other "
      f"non-calls are counted separately above, not folded into this number.")

clinvar_sig (raw, unmodified):
clinvar_sig
Uncertain significance                          2512
Likely benign                                   1536
Pathogenic                                       834
Conflicting classifications of pathogenicity     398
Likely pathogenic                                383
Pathogenic/Likely pathogenic                     256
not provided                                      92
Benign                                            74
Benign/Likely benign                              37
Pathogenic; drug response                         17
drug response                                     12
-                                                  9
no classification for the single variant           4
Pathogenic/Likely pathogenic; other                1

clinsig_simple (ClinVar's own flag: 1=has a pathogenic-like call, 0=no, -1=no data):
clinsig_simple
 0    4412
 1    1740
-1      13

Rows explicitly marked 'Uncertain significance': 2,512 (41% of all rows). 'not p

Notice how **the single largest bucket in ClinVar is `Uncertain significance`**. That is not a bug — it is the *reason predictors exist*. If every variant already had a confident Pathogenic/Benign label, we would not need EVE or AlphaMissense. The huge pile of VUS is precisely the set of variants a lab gets back from sequencing and *cannot act on*. A good predictor's job is to help **re-classify** those VUS — so they are the interesting, unsolved cases, not the easy ones.

## 4 · What kind of variants does ClinVar track?

`variant_type` is ClinVar's own **structural** classification — single
nucleotide variant, Deletion, Insertion, Indel, Duplication, Microsatellite,
Inversion. It does **not** tell you the *molecular consequence* (missense vs.
nonsense vs. splice vs. synonymous) — ClinVar's tab-delimited release doesn't
carry that as its own column. The cell below shows the structural breakdown
first, then derives a coarse functional class from the HGVS `Name` field
(`NM_000492.4(CFTR):c.1521_1523del (p.Phe508del)`-style strings).

The derivation is a short ordered rule set, applied to the cDNA and protein
parts of the HGVS name — **order matters**, because a `delins` can look like a
missense at the protein level:

1. `del` / `ins` / `dup` in the cDNA name, or repeat notation like `ATC[1]` → **indel / large del**
2. `Ter` or a trailing `*` in the protein part → **nonsense**
3. `fs` → **frameshift**
4. a clean `Xxx###Yyy` protein change → **missense**
5. `=` (protein unchanged) → **synonymous**
6. an intronic offset like `c.1234+5` or `c.1234-26` → **intronic / splice-region**
7. `c.-` → **5' UTR**; `c.*` → **3' UTR**
8. no transcript-relative HGVS at all → **genomic-only** (an honest bucket, not a catch-all)

Applying the *same* rules to CFTR2's HGVS names is what lets the two truth sets
be compared without a classification mismatch masquerading as a disagreement.

In [6]:
import re

print("ClinVar's own structural Type:")
print(clinvar['variant_type'].value_counts().to_string())


def clinvar_variant_class(name) -> str:
    """Coarse molecular-consequence class from a ClinVar HGVS Name string, e.g.
    'NM_000492.4(CFTR):c.1521_1523del (p.Phe508del)'. Implements the ordered rule
    set described in the markdown above, adapted to ClinVar's single combined
    Name field (CFTR2 ships cdna_name/protein_name as separate columns; ClinVar
    doesn't). Apply the same rules to CFTR2 to keep the two truth sets
    classified consistently."""
    if not isinstance(name, str):
        return "genomic-only (no transcript HGVS)"
    cdna_m = re.search(r":(c\.[^\s(]+)", name)
    cdna = cdna_m.group(1) if cdna_m else ""
    prot_m = re.search(r"\(p\.([^)]+)\)", name)
    prot = prot_m.group(1) if prot_m else ""
    # indel first -- a delins can also look like a missense/nonsense at the protein
    # level, and repeat-contraction notation (e.g. 'ATC[1]') has no literal 'del'.
    if re.search(r"(del|ins|dup)", cdna) or re.search(r"[ACGT]{2,}\[\d+\]", cdna):
        return "indel / large del"
    if "Ter" in prot or prot.endswith("*"):
        return "nonsense"
    if "fs" in prot:
        return "frameshift"
    if re.match(r"^[A-Z][a-z]{2}\d+[A-Z][a-z]{2}$", prot):
        return "missense"
    if "=" in prot or cdna.rstrip().endswith("="):
        return "synonymous"
    if re.search(r"c\.[\d*-]+[+-]\d+", cdna):
        return "intronic / splice-region"
    if re.match(r"^c\.-", cdna):
        return "5' UTR"
    if re.match(r"^c\.\*", cdna):
        return "3' UTR"
    if not cdna:
        # some large structural variants have no transcript-relative HGVS at all
        # (e.g. 'NC_000007.14:g.117610519_117610669del') -- genuinely not classifiable
        # without picking a transcript, so this is an honest bucket, not a catch-all.
        return "genomic-only (no transcript HGVS)"
    return "other / complex"


clinvar['variant_class'] = clinvar['hgvs_name'].apply(clinvar_variant_class)
print("\nDerived molecular-consequence class (from hgvs_name):")
print(clinvar['variant_class'].value_counts().to_string())

ClinVar's own structural Type:
variant_type
single nucleotide variant    5201
Deletion                      590
Duplication                   174
Indel                          83
Microsatellite                 76
Insertion                      38
Variation                       2
Inversion                       1

Derived molecular-consequence class (from hgvs_name):
variant_class
missense                             2626
synonymous                           1105
intronic / splice-region             1026
indel / large del                     936
nonsense                              315
5' UTR                                 85
genomic-only (no transcript HGVS)      40
3' UTR                                 32


One value in `clinvar_sig` deserves a specific caution if you ever build your own pathogenic/benign call downstream:

> **`Conflicting classifications of pathogenicity`** — labs *disagreed* on this variant.

Do **not** silently fold these into `uncertain` (or worse, resolve them to whichever side has more submitters) without saying so out loud. Why:

- If you force a conflicting variant into `pathogenic` or `benign`, you are inventing a ground-truth label that **doesn't exist** — the experts themselves couldn't agree.
- Scoring a predictor against a made-up label makes the benchmark look cleaner than reality and can unfairly reward *or* punish the tool.

`clinvar_sig` is exposed raw specifically so you can see exactly which rows these are and decide deliberately — keeping them in their own bucket, rather than merging them into VUS, is usually the honest choice.

## 5 · When did ClinVar first call each variant pathogenic?

Everything above describes ClinVar **as it stands today**. That is not enough to benchmark
a predictor against it. **Temporal leakage** is the reason: a variant whose pathogenic
status was public years before a model was trained is in that model's training data, so
scoring it correctly demonstrates recall, not skill. "The tool got this right" and "the
tool memorised it" are the same observation unless you know *when* the call was made.

The fetch cell at the top already knows ClinVar publishes monthly dated snapshots — it
uses them to pin a run. This section uses them for something else: read a series of
snapshots and the history falls out.

### What "the date" means here

`variant_summary` is a snapshot, not a history. It has no "first asserted" field, and its
`LastEvaluated` column is the *most recent* review — which points the wrong way in time.
So the date used here is derived, and worth being exact about:

> **the first archived release whose aggregate classification for this variant is a clean
> Pathogenic / Likely pathogenic / Pathogenic-Likely pathogenic call.**

`Conflicting` is deliberately **not** counted as pathogenic. §4 argues you should not
resolve a disagreement into a call you invent; the same applies here, and counting
conflicts would inflate every hold-out below.

### Sampling, and the floor

Monthly across eleven years is ~140 releases and roughly 20 GB. This reads **one release
per year** instead — the same granularity as `toolkit.TOOL_YEAR`, which records tool
release *years*, so a finer sample would not sharpen any hold-out this feeds. A variant
dated to a release became pathogenic sometime in the preceding twelve months.

The series is **left-censored at its first release**: anything already Pathogenic then
carries a bound, not a date. F508del is one of those — it was reported in 1989, and no
amount of reading the archive recovers that.

### Reproducibility

Every release is public, permanent and unauthenticated at NCBI's FTP archive. The build
cell streams each one, filters to CFTR/GRCh38 on the way in, and caches the result — a few
hundred KB per release instead of the 8 MB–421 MB raw file. First run takes a few minutes
and moves ~1.7 GB; after that it is instant, and an interrupted run resumes.

In [7]:
import clinvar_history as chist

CLINVAR_HIST_CSV = DATA_DIR / "clinvar_history.csv"
CLINVAR_HIST_CACHE = DATA_DIR / "clinvar_history"

if CLINVAR_HIST_CSV.exists():
    print(f"already built -> {CLINVAR_HIST_CSV.name} "
          f"(delete it and clinvar_history.release.json to rebuild)")
else:
    print(f"fetching {len(chist.RELEASES)} archived releases "
          f"({chist.RELEASES[0]} .. {chist.RELEASES[-1]}); each is streamed, filtered to "
          "CFTR/GRCh38 and cached, so this is slow once and instant afterwards")
    paths = chist.fetch_all(CLINVAR_HIST_CACHE)

    hist = chist.build_history(paths)
    meta = hist["meta"]
    print(f"\nreleases chained: {meta['release_count']}  "
          f"({meta['floor_release']} -> {meta['latest_release']})")
    print(f"join key        : {meta['join_key']}")
    print("\nrelease-to-release churn:")
    for step in meta["steps"]:
        print(f"  {step}")

    paths_out = chist.write_extracts(hist, DATA_DIR)
    print(f"\nwrote {paths_out['long'].name} ({len(hist['long']):,} variant x release rows), "
          f"{paths_out['summary'].name} ({len(hist['summary']):,} variants) "
          f"and {paths_out['release'].name}")

CACHED = sorted(CLINVAR_HIST_CACHE.glob("clinvar_cftr_*.tsv"))

fetching 12 archived releases (2015-12 .. 2026-08); each is streamed, filtered to CFTR/GRCh38 and cached, so this is slow once and instant afterwards



releases chained: 12  (2015-12 -> 2026-08)
join key        : allele_id

release-to-release churn:
  2015-12 -> 2016-12: 1,235 -> 1,250 rows (+24 new, -9 gone, of which 8 were classified)
  2016-12 -> 2017-12: 1,250 -> 1,397 rows (+153 new, -6 gone, of which 3 were classified)
  2017-12 -> 2018-12: 1,397 -> 1,494 rows (+341 new, -244 gone, of which 10 were classified)
  2018-12 -> 2019-12: 1,494 -> 1,775 rows (+281 new, -0 gone, of which 0 were classified)
  2019-12 -> 2020-12: 1,775 -> 2,204 rows (+431 new, -2 gone, of which 2 were classified)
  2020-12 -> 2021-12: 2,204 -> 2,612 rows (+408 new, -0 gone, of which 0 were classified)
  2021-12 -> 2022-12: 2,612 -> 3,744 rows (+1,141 new, -9 gone, of which 9 were classified)
  2022-12 -> 2023-12: 3,744 -> 4,261 rows (+526 new, -9 gone, of which 9 were classified)
  2023-12 -> 2024-12: 4,261 -> 4,951 rows (+690 new, -0 gone, of which 0 were classified)
  2024-12 -> 2025-12: 4,951 -> 5,765 rows (+814 new, -0 gone, of which 0 were classifie

### Why the join key is `#AlleleID`, not `VariationID`

Reading twelve releases is a join, and the join is where this goes wrong quietly. Three
things in the archive would each return a plausible wrong answer rather than an error, so
`clinvar_history.py` guards all three:

| what changes | what it would do undetected | guard |
|---|---|---|
| **`VariationID` starts at 2018-12** — the key `load_clinvar()` uses, and the obvious thing to join on | the walk silently starts in 2018 and dates every variant known before then to 2018 | key must exist and be populated in **every** release — raises |
| **the column set grows 25 → 43** across the series | positional indexing reads the wrong column in eleven of twelve releases | every column located by header name; a missing one raises |
| **classification vocabulary drift** — `Conflicting interpretations of pathogenicity` → `Conflicting classifications of pathogenicity`; `no interpretation for the single variant` → `no classification…` | a pure rename reads as a reclassification for every affected variant | mapped to one canonical label, raw string kept |

`#AlleleID` is present and populated in all twelve, so it is the key; the summary carries
`variation_id` from the newest release so the result still joins to `load_clinvar()`.

**A fourth thing is not a trap but looks like one.** Between 2017-12 and 2018-12, 244 CFTR
records vanish. That is not a broken key: **96% of them carry no classification at all**,
ClinVar reorganised which unclassified records `variant_summary` lists, and 139 return in
later releases with the *same* AlleleID. So the dropout guard measures **classified**
records only — the population a date can exist for. On those, the worst step in the whole
series loses 1.78%, and a genuine key break would be far larger.

The cell below runs the build keyed on `VariationID` on purpose, to show the guard firing
on the failure it exists to catch rather than merely asserting that it would.

In [8]:
# NEGATIVE CONTROL -- this build is SUPPOSED to fail.
try:
    chist.build_history(CACHED, key_by="variation_id")
    print("NO GUARD FIRED -- the negative control did not fail, which means the guard "
          "is no longer protecting anything. Do not trust the dates.")
except chist.HistoryError as e:
    print("guard fired as intended, keyed on VariationID:\n")
    print(" ", e)

# ...and the same build on the real key must succeed, or the comparison proves nothing.
chist.build_history(CACHED)
print("\nsame twelve releases keyed on #AlleleID: builds cleanly.")

guard fired as intended, keyed on VariationID:

  'variation_id' is missing or empty in 3 of 12 releases (2015-12, 2016-12, 2017-12). Keying the walk on it would silently start the series at 2018-12 and date every variant known before then to that release.



same twelve releases keyed on #AlleleID: builds cleanly.


### The output: one row per variant per release

The build writes `data/clinvar_history_long.csv` — the actual result of this section,
keyed on **(variant, release)** with ClinVar's classification at that release:

| column | |
|---|---|
| `allele_id` | ClinVar's `#AlleleID` — the only identifier present in all twelve releases |
| `release` | which archived release this row describes |
| `significance` | ClinVar's exact aggregate string at that release |
| `significance_canon` | the same, normalised (see below) |
| `is_pathogenic` | whether the canonical label is a clean pathogenic call |
| `review_status` | the star rating at that release — it moves too |

Both the raw string and the normalised form are kept, because normalising ClinVar's
significance field is a **real collapse and a judgement call**, and §3 argues against
hiding one. Two rules do the work, and neither invents a clinical opinion:

- **Renames are merged** — `Conflicting interpretations…` and `Conflicting
  classifications…` are the same label under two names.
- **Pre-2017 rows are un-collapsed first.** In 2015–2016 ClinVar joined *disagreeing
  submitter calls* with `;` — `Benign;Likely benign;Pathogenic;Uncertain significance` is
  one row reporting four different opinions. From 2017 it replaced that with the single
  `Conflicting…` label and reused `;` for trailing modifiers (`Pathogenic; drug
  response`). Reading only the first token would score that 2015 four-way disagreement as
  a clean `Pathogenic` and date the variant years too early — so each part is parsed, and
  a genuine disagreement becomes `conflicting`. Concordant pairs (`Likely
  pathogenic;Pathogenic`) become the combined label ClinVar itself later adopted.

`data/clinvar_history.csv` sits beside it with per-variant summaries for the hold-out.
Both are gitignored — ClinVar is public domain and *could* ship, but these are bulky and
rebuilt by running the cell above; `data/publishable/` carries the current snapshot.

In [9]:
cv_long = pd.read_csv(DATA_DIR / "clinvar_history_long.csv", dtype={"allele_id": "string"})
cv_hist = tk.load_clinvar_history()

print(f"table shape      : {len(cv_long):,} rows x {cv_long.shape[1]} columns "
      "(one row per variant per release)")
print(f"releases covered : {cv_long['release'].nunique()} "
      f"({cv_long['release'].min()} -> {cv_long['release'].max()})")
print(f"distinct variants: {cv_long['allele_id'].nunique():,} "
      f"({int(cv_hist['clinvar_in_current_release'].sum()):,} in the newest release)")

changed = cv_hist["clinvar_class_changes"] > 0
ever_p = cv_hist["clinvar_first_pathogenic"].notna()
print(f"\ntotal variants tracked                  : {len(cv_hist):,}")
print(f"ever called pathogenic                  : {int(ever_p.sum()):,}")
print(f"changed classification at least once    : {int(changed.sum()):,} "
      f"({changed.mean():.2%})")
print(f"  were pathogenic and later were not    : {int(cv_hist['clinvar_ever_withdrawn'].sum()):,}")

print("\nfirst called pathogenic, by release "
      f"(the {cv_hist['clinvar_first_pathogenic'].eq(cv_long['release'].min()).sum()} at "
      f"{cv_long['release'].min()} are a bound, not a date):")
print(cv_hist["clinvar_first_pathogenic"].value_counts().sort_index().to_string())

table shape      : 36,862 rows x 7 columns (one row per variant per release)
releases covered : 12 (2015-12 -> 2026-08)
distinct variants: 6,309 (6,174 in the newest release)

total variants tracked                  : 6,309
ever called pathogenic                  : 1,618
changed classification at least once    : 1,246 (19.75%)
  were pathogenic and later were not    : 122

first called pathogenic, by release (the 351 at 2015-12 are a bound, not a date):
clinvar_first_pathogenic
2015-12    351
2016-12      6
2017-12     59
2018-12    125
2019-12    164
2020-12    193
2021-12     49
2022-12    217
2023-12     69
2024-12    135
2025-12    107
2026-08    143


### What this buys: a hold-out per tool

For each predictor, the variants ClinVar first called pathogenic **after** that tool was
released could not have been in its training data as pathogenic. Those are the ones where
a correct score is evidence of skill rather than recall.

`TOOL_YEAR` records release years and this series is sampled annually, so the two line up
directly. A variant counts only if its first pathogenic release falls in a year *strictly
after* the tool's — same-year cases are excluded rather than guessed at.

`LABEL_SUPERVISED` marks the tools trained directly on curated clinical labels. For those,
leakage is a first-order problem and this table is the fix. The rest never saw clinical
labels at all and can only leak indirectly, through the literature that informed them.

In [10]:
cur = cv_hist[cv_hist["clinvar_in_current_release"]]          # the truth set as it stands
first_p = cur["clinvar_first_pathogenic"]
# Anything already pathogenic at the first archived release carries a bound, not a date,
# so it can never count toward a hold-out -- which is the safe direction to err in.
floor = cv_long["release"].min()
datable_year = first_p[first_p.notna() & (first_p != floor)].str.slice(0, 4).astype(int)

holdout = pd.DataFrame(
    [{"tool": t,
      "released": yr,
      "label_supervised": tk.LABEL_SUPERVISED[t],
      "pathogenic_after_release": int((datable_year > yr).sum())}
     for t, yr in sorted(tk.TOOL_YEAR.items(), key=lambda kv: kv[1])])
print(holdout.to_string(index=False))

negatives = int((~cur["clinvar_current_significance"].isin(sorted(chist.PATHOGENIC))).sum())
print()
print(f"candidate negatives available to every tool: {negatives:,} variants ClinVar does "
      "not currently call pathogenic")
print(f"of which 'conflicting'                     : "
      f"{int((cur['clinvar_current_significance'] == 'conflicting').sum()):,} "
      "-- excluded from BOTH sides; labs disagree, so there is no label to score against")

         tool  released  label_supervised  pathogenic_after_release
        REVEL      2016              True                      1254
    PrimateAI      2018             False                      1073
     SpliceAI      2019             False                       910
          EVE      2021             False                       671
     Pangolin      2022             False                       454
AlphaMissense      2023             False                       385
        ESM1b      2023             False                       385
         CADD      2024             False                       250

candidate negatives available to every tool: 4,680 variants ClinVar does not currently call pathogenic
of which 'conflicting'                     : 398 -- excluded from BOTH sides; labs disagree, so there is no label to score against


### What this date can and cannot support

**It can** separate recall from skill for any tool released after the archive opens, and
it says which variants to use: score only the ones ClinVar first called pathogenic after
the tool shipped, against the ones it does not call pathogenic as negatives.

**It cannot** tell you when a variant was *discovered*, or when anyone first called it
disease-causing. It measures one thing — when **ClinVar's aggregate** first read
pathogenic. Three gaps follow, and none is closed by reading more releases:

- **Everything at the first archived release is one bucket**, ordered relative to nothing.
  F508del and a variant first submitted in 2014 are indistinguishable.
- **Resolution is one year**, by choice of sampling.
- **ClinVar is a follower.** A variant is typically reported in the literature well before
  a lab submits it, so the date is an **upper bound** on when the label became public.
  For a hold-out that errs the safe way: it can wrongly exclude a variant as leaked, but it
  will not wrongly admit one.

### ClinVar moves; CFTR2 does not

Run the same reconstruction on CFTR2 (benchmark/01 §2) and the contrast is the useful
result. Roughly **one in five** ClinVar variants has changed classification at least once,
and over a hundred were pathogenic and later were not. The equivalent CFTR2 numbers are
**1%** and **one variant**.

That is a difference in kind, not in degree, and it is what §2's star ratings predict:
ClinVar is an open ledger where a single submitter can post a call and later revise it,
while CFTR2 publishes an expert-panel determination on a slow cycle. Neither is "better" —
but a benchmark built on ClinVar is standing on ground that moves, and a result that would
change if you had run it two years earlier should be reported as such.

## Key takeaways

1. **ClinVar** aggregates clinical assertions from many submitters; **review status** (the gold stars) tells you how much to trust each one.
2. This notebook keeps **every** CFTR/GRCh38 row ClinVar has (6,000+), not just the missense subset — `protein_variant` is `''` for anything that isn't a simple missense change, so nothing is silently dropped.
3. `clinvar_sig` (raw text) and `clinsig_simple` (ClinVar's own binary pathogenic-like flag) are exposed **unmodified**, with no collapsed pathogenic/benign/uncertain call layered on top — collapsing categorical clinical text hides real distinctions (e.g. "not provided" is not the same thing as "assessed and uncertain").
4. `variant_type` gives ClinVar's own structural classification (SNV/indel/…); a derived `variant_class` (missense/nonsense/splice/etc., from `hgvs_name`) applies the ordered rule set in §4 — the same rules should be applied to CFTR2 so the two truth sets are classified consistently.
5. **Version control:** the fetch cell records a real timestamp (or a pinned `YYYY-MM`) into `clinvar_release` on every row, so every table downstream carries its own provenance.
6. **§5 reconstructs what ClinVar called each variant at each of twelve archived releases**, 2015-12 → 2026-08, into `data/clinvar_history_long.csv`. `variant_summary` has no "first asserted" field and its `LastEvaluated` points the wrong way in time, so the history has to be read out of the archive. That table is what a training-cutoff hold-out is built from.
7. **ClinVar's classifications move.** About **one in five** variants has changed classification at least once and 122 were pathogenic and later were not — against 1% and one variant for CFTR2. A benchmark built on ClinVar rests on ground that shifts, and the date is an **upper bound** on when a label became public.

**Next:** benchmark/01 — **CFTR2**, built from patient data and functional assays.